## Script extracting panicle height and width from prediction and best model is from FN field images

In [1]:
import os
# os.chdir("../../") # Set the relative path for loading the data correctly (must run only once)
os.getcwd()

'c:\\GitHub_AI_Workshop\\codes\\2_panicle_area\\1_area_from_predicted_bbox_pixel'

In [2]:
import numpy as np # For taking random number
import pandas as pd
from ultralytics import YOLO
np.random.seed(33) # For having same random sequence
# home = os.getcwd() # home directory
# home

In [3]:
# Create an empty DataFrame
df = pd.DataFrame(columns=['id', 'image_name', 'rep', 'height_pixels', 'width_pixels'])

# Set the folder path
id_folder = os.path.join("..", "..", "..", "sample_data", "annotated_panicles_lab")

# Prepare data for the DataFrame
data = []
for root, dirs, files in os.walk(id_folder):
    folder_name = os.path.basename(root)
    r = 1
    # Filter only JPG images
    images = [f for f in files if f.lower().endswith('jpg')]
    if not images:
        continue  # Skip if no images are found
    for file in images:
        data.append([folder_name, file, r])
        r += 1

# Convert the data into a DataFrame
if data:  # Ensure there is data to add
    df_temp = pd.DataFrame(data, columns=['id', 'image_name', 'rep'])
    df = pd.concat([df, df_temp], ignore_index=True)

# Filter out rows where 'id' equals 'panicle_size_lab'
df = df[df['id'] != os.path.basename(id_folder)].reset_index(drop=True)

# Display the DataFrame
df

,id,image_name,rep,height_pixels,width_pixels
0,2005-SC964,1. P1-WOF-Fr.jpg,1,NaN,NaN
1,2005-SC964,11. P3-WOF-Ba.jpg,2,NaN,NaN
2,2005-SC964,13. P4-WOF-Fr.jpg,3,NaN,NaN
3,2005-SC964,15. P4-WOF-Ba.jpg,4,NaN,NaN
4,2005-SC964,3. P1-WOF-Ba.jpg,5,NaN,NaN
5,2005-SC964,5. P2-WOF-Fr.jpg,6,NaN,NaN
6,2005-SC964,7. P2-WOF-Ba.jpg,7,NaN,NaN
7,2005-SC964,9. P3-WOF-Fr.jpg,8,NaN,NaN
8,2009-RTx430BL,1. P1-WOF-Fr.jpg,1,NaN,NaN
9,2009-RTx430BL,11. P3-WOF-Ba.jpg,2,NaN,NaN


In [4]:
weight = os.path.join("..", "..", "..", "weights", "panicle_area", "yolov8_fn_best.pt")
weight

'..\\..\\..\\weights\\panicle_area\\yolov8_fn_best.pt'

In [5]:
model = YOLO(weight)  # load a custom model

for root, dirs, files in os.walk(id_folder):
    folder_name = os.path.basename(root)
    if folder_name == os.path.basename(id_folder):
        continue
    image_folder_path = os.path.join(id_folder, folder_name)
    images = [f for f in files if f.lower().endswith('jpg')]  # Filter images
    if not images:
        continue # if there are no image in the folder, then continue loop
    
    print(images)
    
    # Process each image-annotation pair
    for img_i in range(len(images)):
        rand_image = images[img_i]
        rand_image_path = os.path.join(image_folder_path, rand_image)
        results = model(rand_image_path, save=True, imgsz=640, max_det = 30, conf = 0.25, show_labels=False, show_conf=False, project=r'Panicle_Area\area_from_predicted_bboxes\outputs', name=f'{folder_name}_{img_i}') # predict on an image
        xm, ym, width_pixels, height_pixels = results[0].boxes.xywh[0].cpu().numpy()
        print(height_pixels, width_pixels)
        df.loc[(df['id'] == folder_name) & (df['image_name'] == rand_image),
                ['height_pixels', 'width_pixels']] = [height_pixels, width_pixels]

['1. P1-WOF-Fr.jpg', '11. P3-WOF-Ba.jpg', '13. P4-WOF-Fr.jpg', '15. P4-WOF-Ba.jpg', '3. P1-WOF-Ba.jpg', '5. P2-WOF-Fr.jpg', '7. P2-WOF-Ba.jpg', '9. P3-WOF-Fr.jpg']

image 1/1 c:\GitHub_AI_Workshop\codes\2_panicle_area\1_area_from_predicted_bbox_pixel\..\..\..\sample_data\annotated_panicles_lab\2005-SC964\1. P1-WOF-Fr.jpg: 448x640 3 panicles, 24.0ms
Speed: 2.3ms preprocess, 24.0ms inference, 13.7ms postprocess per image at shape (1, 3, 448, 640)
Results saved to C:\GitHub_AI_Workshop\codes\2_panicle_area\1_area_from_predicted_bbox_pixel\runs\detect\Panicle_Area\area_from_predicted_bboxes\outputs\2005-SC964_0
234.12183 387.75696

image 1/1 c:\GitHub_AI_Workshop\codes\2_panicle_area\1_area_from_predicted_bbox_pixel\..\..\..\sample_data\annotated_panicles_lab\2005-SC964\11. P3-WOF-Ba.jpg: 448x640 2 panicles, 9.9ms
Speed: 2.1ms preprocess, 9.9ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
Results saved to C:\GitHub_AI_Workshop\codes\2_panicle_area\1_area_from_predicted_

In [6]:
df

,id,image_name,rep,height_pixels,width_pixels
0,2005-SC964,1. P1-WOF-Fr.jpg,1,234.121826,387.756958
1,2005-SC964,11. P3-WOF-Ba.jpg,2,1855.050537,450.019775
2,2005-SC964,13. P4-WOF-Fr.jpg,3,1633.118408,680.255127
3,2005-SC964,15. P4-WOF-Ba.jpg,4,1624.055664,648.188477
4,2005-SC964,3. P1-WOF-Ba.jpg,5,1586.657715,637.837402
5,2005-SC964,5. P2-WOF-Fr.jpg,6,1441.640381,593.094238
6,2005-SC964,7. P2-WOF-Ba.jpg,7,226.550659,393.854736
7,2005-SC964,9. P3-WOF-Fr.jpg,8,1856.998779,501.238037
8,2009-RTx430BL,1. P1-WOF-Fr.jpg,1,2230.703125,556.394531
9,2009-RTx430BL,11. P3-WOF-Ba.jpg,2,1836.011597,506.297363


In [7]:
csv_path = os.path.join(
    os.getcwd(),
    "height_and_widths_predicted_panicle.csv"
)

df.to_csv(csv_path, index=False)

print(f"CSV saved successfully at: {csv_path}")

CSV saved successfully at: c:\GitHub_AI_Workshop\codes\2_panicle_area\1_area_from_predicted_bbox_pixel\height_and_widths_predicted_panicle.csv
